# Week 7 — Designing a fair experiment and freezing the study

## Overview

We now turn the earlier demonstrations into a small research study. The question is:
**at a fixed total simulation budget, how do representative selection, value prediction,
and learned similarity change the accuracy of a GMAB pricing approximation?**

This notebook is the final **development** stage. All model choices use simulated
validation labels. The held-out test contracts remain unopened until Week 8. The
notebook runs independently: its full pricing and learning code is included below.

**Learning objectives**

1. Define what “the same computation budget” means and distinguish it from runtime.
2. Change one component at a time while keeping labels and other conditions comparable.
3. Record model selection, random seeds, simulation work and runtime honestly.
4. Lock a design before inspecting final test performance.

**Prerequisites:** Weeks 1–6, sample means and variances, train/validation/test splits.
**Weekly effort:** one 75-minute meeting and approximately 4–6 hours of independent work.

**By the end:** a validation comparison table, a simulation-work ledger, an accuracy–cost
plot, and a JSON freeze manifest. No final model ranking is promised.

## 1. The question determines the experiment

The outcome is the per-premium expected discounted **gross GMAB top-up**, $y(x)$, at a
given fee. We assume survival to maturity and do not model mortality, lapses or
withdrawals. Dollar liability is $P_x y(x)$. The fee is an input, not a root to solve.

We separate three questions:

| Component | Controlled change | What remains fixed within a comparison |
|---|---|---|
| Compressor | Random representatives versus K-means member representatives | Candidate pool, label budget, predictor family |
| Predictor | CSM-style 1-NN, k-NN, ridge, additive spline regression, random forest | Representatives and the exact MC label realizations |
| Similarity | Equal feature weights versus learned nonnegative weights | Representatives, path counts and one-neighbor transfer rule |

Gradient boosting is supplied as an optional additional predictor. A random forest
that directly predicts values does not by itself learn a distance. K-means coverage
and valuation accuracy are also different quantities.

The first experiment should explain these differences, not try every available model.
The supplied grid is small; its loops automate work already introduced in earlier weeks.

## 2. Equal simulation work is a design choice

Let $B$ be the maximum number of individual contract-path payoff evaluations available
to one complete approximation pipeline. Reserve $n_v M_v$ of them for validation:

$$B_v=n_vM_v,\qquad M_r(m)=\left\lfloor\frac{B-B_v}{m}\right\rfloor.$$

Each of $m$ representatives receives $M_r(m)$ paths. Increasing the number of
representatives improves coverage but reduces the precision of each label.
The direct-all Monte Carlo baseline receives

$$M_{\mathrm{all}}=\left\lfloor\frac{B}{N}\right\rfloor$$

paths for **every one of the $N$ contracts**. It does not need validation labels or model
selection. All MC estimates require at least two paths. Integer rounding leaves a
small, explicitly reported unused remainder.

In this teaching model one path uses one normal draw and one terminal payoff,
irrespective of maturity. That makes path counts a useful work measure. They would
need adjustment for time-stepped or nested models.

There are two legitimate but distinct studies:

* **Equal total work:** compare error and measured runtime under the same budget ceiling.
* **Equal paths per simulated contract:** compare actual total work and accuracy when
  direct MC uses $NM$ paths and a representative method uses roughly $mM+B_v$.

The final core experiment uses the first design. Do not describe results from the
second design as a fixed-budget comparison.

In [ ]:
# Complete common pricing code. No data files or earlier notebooks are required.
import sys
import json
import platform
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

try:
    from IPython.display import display
except ImportError:
    def display(*objects):
        for obj in objects:
            print(obj)

RISK_FREE_RATE = 0.03
FEATURES = ['account_ratio', 'guarantee_ratio', 'maturity', 'volatility', 'fee']
plt.rcParams.update({'figure.figsize': (7.2, 4.3), 'axes.grid': True,
                     'grid.alpha': 0.22, 'font.size': 10})


def make_portfolio(n=1200, seed=2027):
    """Synthetic in-force GMAB contracts, never customer data.

    P is a reference premium, A0=P*account_ratio, and G=P*guarantee_ratio.
    T is the remaining term. All contracts share the same risk-free rate.
    Volatility describes the contract's fund; rates are annual decimals.
    """
    if not isinstance(n, (int, np.integer)) or n < 10:
        raise ValueError('n must be an integer of at least 10.')
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        'contract_id': np.arange(n, dtype=int),
        'premium': rng.uniform(10_000.0, 50_000.0, n),
        'account_ratio': rng.uniform(0.65, 1.40, n),
        'guarantee_ratio': rng.uniform(0.90, 1.60, n),
        'maturity': rng.integers(5, 21, n).astype(float),
        'volatility': rng.uniform(0.12, 0.35, n),
        'fee': rng.uniform(0.005, 0.030, n),
    })


def validate_contracts(df):
    required = ['contract_id', 'premium'] + FEATURES
    missing = [c for c in required if c not in df]
    if missing:
        raise ValueError(f'Missing contract columns: {missing}')
    if len(df) == 0:
        raise ValueError('Supply at least one contract.')
    values = df[required].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError('All contract fields must be finite.')
    ids = df['contract_id'].to_numpy(dtype=float)
    if (ids < 0).any() or (ids != np.floor(ids)).any() or len(np.unique(ids)) != len(ids):
        raise ValueError('contract_id must contain unique nonnegative integers.')
    if (df[['premium', 'account_ratio', 'guarantee_ratio']] <= 0).any().any():
        raise ValueError('Premium and account/guarantee ratios must be positive.')
    if (df[['maturity', 'volatility', 'fee']] < 0).any().any():
        raise ValueError('Maturity, volatility and fee must be nonnegative.')


def gmab_mc(df, n_paths=2048, seed=123):
    """Independent plain Monte Carlo for per-premium gross top-up liability.

    Identical (seed, contract_id) reproduces the same path stream even when
    the contract appears in a different subset or row order. Distinct IDs
    give independent streams. This is NOT a common-market portfolio-risk
    simulator: it estimates a sum of expected liabilities, not portfolio VaR.
    SE describes Monte Carlo sampling uncertainty only.
    """
    validate_contracts(df)
    if not isinstance(n_paths, (int, np.integer)) or n_paths < 2:
        raise ValueError('n_paths must be an integer of at least 2.')
    if not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('seed must be a nonnegative integer.')
    values, errors, sds = [], [], []
    for row in df.itertuples():
        rng = np.random.default_rng(np.random.SeedSequence([int(seed), int(row.contract_id)]))
        z = rng.standard_normal(n_paths)
        terminal_ratio = row.account_ratio * np.exp(
            (RISK_FREE_RATE - row.fee - 0.5 * row.volatility**2) * row.maturity
            + row.volatility * np.sqrt(row.maturity) * z
        )
        payoff = np.exp(-RISK_FREE_RATE * row.maturity) * np.maximum(
            row.guarantee_ratio - terminal_ratio, 0.0)
        sd = float(np.std(payoff, ddof=1))
        values.append(float(np.mean(payoff)))
        errors.append(sd / np.sqrt(n_paths))
        sds.append(sd)
    return pd.DataFrame({'value': values, 'se': errors, 'payoff_sd': sds,
                         'paths': int(n_paths)}, index=df.index)


def gmab_exact(df):
    """Closed-form put benchmark under the simplified model: audit only.

    Never use these values as training or tuning labels in the project.
    The deterministic cases T=0 or sigma=0 are handled separately.
    """
    validate_contracts(df)
    a = df['account_ratio'].to_numpy(dtype=float)
    g = df['guarantee_ratio'].to_numpy(dtype=float)
    t = df['maturity'].to_numpy(dtype=float)
    sig = df['volatility'].to_numpy(dtype=float)
    fee = df['fee'].to_numpy(dtype=float)
    ans = np.maximum(g * np.exp(-RISK_FREE_RATE*t) - a * np.exp(-fee*t), 0.0)
    regular = (t > 0) & (sig > 0)
    if np.any(regular):
        tr, sr = t[regular], sig[regular]
        d1 = (np.log(a[regular] / g[regular]) +
              (RISK_FREE_RATE-fee[regular]+0.5*sr**2)*tr) / (sr*np.sqrt(tr))
        d2 = d1 - sr*np.sqrt(tr)
        ans[regular] = (g[regular]*np.exp(-RISK_FREE_RATE*tr)*norm.cdf(-d2)
                        - a[regular]*np.exp(-fee[regular]*tr)*norm.cdf(-d1))
    return np.maximum(ans, 0.0)


def split_portfolio(df, seed=2027):
    """Split contracts before building pairs, selecting labels or tuning."""
    validate_contracts(df)
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(df))
    n_candidate, n_validation = int(0.60*len(df)), int(0.20*len(df))
    if min(n_candidate, n_validation, len(df)-n_candidate-n_validation) < 1:
        raise ValueError('Portfolio is too small for the requested split.')
    return {
        'candidate': df.iloc[order[:n_candidate]].copy(),
        'validation': df.iloc[order[n_candidate:n_candidate+n_validation]].copy(),
        'test': df.iloc[order[n_candidate+n_validation:]].copy(),
    }


def score_predictions(y_true, y_pred, premium=None):
    """Errors per premium, plus signed aggregate relative error when requested."""
    truth = np.asarray(y_true, dtype=float).reshape(-1)
    pred = np.asarray(y_pred, dtype=float).reshape(-1)
    if truth.shape != pred.shape or len(truth) == 0:
        raise ValueError('Truth and predictions must have equal nonzero length.')
    if not (np.isfinite(truth).all() and np.isfinite(pred).all()):
        raise ValueError('Truth and predictions must be finite.')
    error = pred-truth
    out = {'rmse': float(np.sqrt(np.mean(error**2))),
           'mae': float(np.mean(np.abs(error))),
           'p95_abs': float(np.quantile(np.abs(error), 0.95)),
           'bias': float(np.mean(error)),
           'negative_fraction': float(np.mean(pred < 0))}
    if premium is not None:
        p = np.asarray(premium, dtype=float).reshape(-1)
        if p.shape != truth.shape or not np.isfinite(p).all() or np.any(p <= 0):
            raise ValueError('Premium weights must be finite, positive and aligned.')
        denominator = float(p @ truth)
        out['portfolio_relative_error'] = (float(p @ error)/denominator
                                            if abs(denominator) > 1e-12 else np.nan)
    return out


In [ ]:
# Complete shared learning code. All labels are supplied explicitly by the caller.
from scipy.spatial.distance import cdist
from scipy.optimize import minimize
from sklearn.preprocessing import MinMaxScaler, SplineTransformer
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline


def build_features(candidate, query=None, columns=FEATURES):
    """Fit feature scales only on the unlabelled candidate pool.

    A constant training column maps to zero on training data. A held-out
    observation outside the training range may map outside [0,1]; no clipping.
    No premium, contract ID or computed valuation is an informative feature.
    """
    scaler = MinMaxScaler()
    z_candidate = scaler.fit_transform(candidate[list(columns)])
    z_query = None if query is None else scaler.transform(query[list(columns)])
    return scaler, z_candidate, z_query


def select_representatives(Z, m, method='kmeans', seed=2027):
    """Return m DISTINCT row positions of actual candidate contracts.

    K-means is fitted to candidate FEATURES only. For each nonempty cluster,
    choose the member closest to that cluster's center. This is a member
    representative, not an exact k-medoids optimization or Feng et al.'s SRSC.
    """
    z = np.asarray(Z, dtype=float)
    if z.ndim != 2 or not np.isfinite(z).all():
        raise ValueError('Z must be a finite 2D matrix.')
    if not isinstance(m, (int, np.integer)) or not 1 <= m <= len(z):
        raise ValueError('m must lie between 1 and the number of candidates.')
    rng = np.random.default_rng(seed)
    if method == 'random':
        return np.sort(rng.choice(len(z), size=m, replace=False))
    if method != 'kmeans':
        raise ValueError("method must be 'random' or 'kmeans'.")
    km = KMeans(n_clusters=m, n_init=5, max_iter=150, random_state=seed)
    membership = km.fit_predict(z)
    chosen = []
    for cluster in range(m):
        members = np.flatnonzero(membership == cluster)
        if len(members):
            distances = np.sum((z[members] - km.cluster_centers_[cluster])**2, axis=1)
            chosen.append(int(members[np.argmin(distances)]))
    # Duplicate feature vectors can yield fewer than m nonempty clusters.
    # Keep the requested contract count without ever repeating an ID.
    if len(chosen) < m:
        remaining = np.setdiff1d(np.arange(len(z)), chosen)
        chosen.extend(rng.choice(remaining, m-len(chosen), replace=False).tolist())
    return np.sort(np.asarray(chosen, dtype=int))


def nearest_predict(Zref, yref, Zquery, weights=None, k=1):
    """Uniform k-neighbor transfer; k=1 is the CSM-style teaching baseline.

    A diagonal weighted distance is ordinary Euclidean distance after each
    coordinate is multiplied by sqrt(weight). Stable sort breaks exact ties
    in reference-row order. Chunking avoids an N-by-N distance matrix.
    """
    zr, zq = np.asarray(Zref, dtype=float), np.asarray(Zquery, dtype=float)
    yr = np.asarray(yref, dtype=float).reshape(-1)
    if zr.ndim != 2 or zq.ndim != 2 or zr.shape[1] != zq.shape[1] or len(zr) != len(yr):
        raise ValueError('Reference/query dimensions or labels do not align.')
    if not (np.isfinite(zr).all() and np.isfinite(zq).all() and np.isfinite(yr).all()):
        raise ValueError('Features and labels must be finite.')
    if not isinstance(k, (int, np.integer)) or not 1 <= k <= len(zr):
        raise ValueError('k must be between 1 and the reference count.')
    w = np.ones(zr.shape[1]) if weights is None else np.asarray(weights, dtype=float)
    if w.shape != (zr.shape[1],) or not np.isfinite(w).all() or np.any(w < 0):
        raise ValueError('Distance weights must be finite and nonnegative.')
    scale = np.sqrt(w)
    neighbors = np.empty((len(zq), k), dtype=int)
    for start in range(0, len(zq), 1024):
        stop = min(start+1024, len(zq))
        distance = cdist(zq[start:stop]*scale, zr*scale, metric='sqeuclidean')
        neighbors[start:stop] = np.argsort(distance, axis=1, kind='stable')[:, :k]
    return yr[neighbors].mean(axis=1), neighbors


def fit_metric(Zref, yref, lam=0.01, max_pairs=2000, seed=2027):
    """Fit a simple diagonal response-aware pseudometric on reference pairs.

    Objective: mean(((H@w)/p - ((y_i-y_j)/s)^2)^2)
               + lam/2 * sum((w-1)^2),  w>=0, sum(w)=p.
    Here H=(z_i-z_j)^2 coordinatewise and s=max(ptp(yref),1e-6).
    The label scale is part of the modeling choice, fitted on references only.
    Numerical labels have noise: this loss is not an unbiased exact-gap loss.
    """
    z, y = np.asarray(Zref, dtype=float), np.asarray(yref, dtype=float).reshape(-1)
    if z.ndim != 2 or len(z) != len(y) or len(y) < 2:
        raise ValueError('Supply at least two aligned labeled reference contracts.')
    if not (np.isfinite(z).all() and np.isfinite(y).all()):
        raise ValueError('Training data must be finite.')
    if lam < 0 or not isinstance(max_pairs, (int, np.integer)) or max_pairs < 1:
        raise ValueError('lam must be nonnegative and max_pairs a positive integer.')
    n, p = z.shape
    count = min(max_pairs, n*(n-1)//2)
    rng = np.random.default_rng(seed)
    if count == n*(n-1)//2:
        i, j = np.triu_indices(n, k=1)
    else:
        # Uniform directed non-self pairs sampled WITH replacement.
        # They share contracts and are not independent observational units.
        i = rng.integers(0, n, size=count)
        j = (i+rng.integers(1, n, size=count)) % n
    label_scale = max(float(np.ptp(y)), 1e-6)
    h = (z[i]-z[j])**2 / p
    target = ((y[i]-y[j])/label_scale)**2
    def objective(w):
        residual = h @ w-target
        return float(np.mean(residual**2) + 0.5*lam*np.sum((w-1.0)**2))
    def gradient(w):
        return 2.0*h.T @ (h @ w-target)/len(target) + lam*(w-1.0)
    result = minimize(objective, np.ones(p), jac=gradient, method='SLSQP',
                      bounds=[(0.0, float(p))]*p,
                      constraints=[{'type': 'eq', 'fun': lambda w: w.sum()-p,
                                    'jac': lambda w: np.ones(p)}],
                      options={'maxiter': 400, 'ftol': 1e-10})
    feasible = (np.isfinite(result.x).all() and result.x.min() >= -1e-7
                and abs(result.x.sum()-p) <= 1e-6)
    success = bool(result.success and feasible)
    weights = np.clip(result.x, 0.0, None) if success else np.ones(p)
    weights *= p/weights.sum()
    return {'weights': weights, 'success': success, 'pair_count': len(target),
            'objective': objective(weights), 'scale': label_scale,
            'iterations': int(result.nit), 'message': str(result.message)}


def make_regressors(seed=2027):
    """Fixed small model menu. Regression predicts values, not distances."""
    return {
        'Ridge': Ridge(alpha=1.0),
        'SplineGAM': make_pipeline(
            SplineTransformer(n_knots=4, degree=3, include_bias=False),
            Ridge(alpha=1.0)),
        'RandomForest': RandomForestRegressor(n_estimators=100, min_samples_leaf=3,
                                               random_state=seed, n_jobs=1),
        'GradientBoosting': GradientBoostingRegressor(n_estimators=100, max_depth=2,
                                                       learning_rate=0.05, random_state=seed),
    }


In [ ]:
# Reproducible benchmark helpers; pricing and learning definitions appear above.
import hashlib
import sklearn
import scipy

DEFAULT_PROTOCOL = {
    'n_contracts': 1200, 'portfolio_seed': 101, 'split_seed': 2027,
    'representative_counts': [40, 80], 'compressors': ['random', 'kmeans'],
    'seeds': [2027, 2028], 'budget': 262144,
    'n_validation_labels': 40, 'validation_paths': 1024,
    'families': ['CSM1NN', 'KNN', 'Ridge', 'SplineGAM', 'RandomForest',
                 'GradientBoosting', 'Learned1NN'],
    'extension': 'none', 'test_audit_paths': 8192,
    'label_target': 'normalized_gross_GMAB_top_up_at_fixed_fee',
    'risk_free_rate': 0.03,
}


def budget_plan(n_total, m, budget, n_val, val_paths):
    inputs = [n_total, m, budget, n_val, val_paths]
    if any(not isinstance(x, (int, np.integer)) or x <= 0 for x in inputs):
        raise ValueError('Budget inputs must be positive integers.')
    b_val = n_val*val_paths
    ref_paths = (budget-b_val)//m
    direct_paths = budget//n_total
    if min(ref_paths, direct_paths, val_paths) < 2:
        raise ValueError('Infeasible budget: require >=2 paths for every MC estimate.')
    return {'reference_paths': int(ref_paths), 'direct_paths': int(direct_paths),
            'reference_work': int(m*ref_paths), 'validation_work': int(b_val),
            'pipeline_work': int(m*ref_paths+b_val),
            'direct_work': int(n_total*direct_paths),
            'pipeline_unused': int(budget-m*ref_paths-b_val),
            'direct_unused': int(budget-n_total*direct_paths)}


def _regression_candidates(family, seed):
    if family == 'Ridge':
        return [(f'alpha={a:g}', Ridge(alpha=a)) for a in [0.1, 1.0, 10.0]]
    if family == 'SplineGAM':
        return [(f'knots=4;alpha={a:g}', make_pipeline(
            SplineTransformer(n_knots=4, degree=3, include_bias=False), Ridge(alpha=a)))
                for a in [0.1, 1.0, 10.0]]
    if family == 'RandomForest':
        return [(f'100trees;leaf={leaf}', RandomForestRegressor(
            n_estimators=100, min_samples_leaf=leaf, random_state=seed, n_jobs=1))
                for leaf in [2, 5]]
    if family == 'GradientBoosting':
        return [('100trees;depth=2;lr=.05', make_regressors(seed)[family])]
    raise ValueError(f'Unknown regression family: {family}')


def develop_one(df, protocol, m, compressor, seed):
    """Fit/tune only on candidate labels and validation labels. No test truth."""
    t0 = perf_counter()
    parts = split_portfolio(df, seed=int(protocol['split_seed']))
    candidate = parts['candidate']
    n_val = int(protocol['n_validation_labels'])
    if n_val > len(parts['validation']):
        raise ValueError('Not enough validation contracts in the fixed split.')
    validation = parts['validation'].iloc[:n_val].copy()
    scaler, z_candidate, z_val = build_features(candidate, validation)
    prep_seconds = perf_counter()-t0
    work = budget_plan(len(df), int(m), int(protocol['budget']), n_val,
                       int(protocol['validation_paths']))
    t0 = perf_counter()
    positions = select_representatives(z_candidate, m, method=compressor, seed=seed)
    representatives = candidate.iloc[positions]
    zr = z_candidate[positions]
    compression_seconds = perf_counter()-t0
    t0 = perf_counter()
    reference_labels = gmab_mc(representatives, work['reference_paths'], seed=seed+10000)
    reference_seconds = perf_counter()-t0
    t0 = perf_counter()
    validation_labels = gmab_mc(validation, int(protocol['validation_paths']), seed=seed+20000)
    validation_seconds = perf_counter()-t0
    yr = reference_labels['value'].to_numpy()
    yv = validation_labels['value'].to_numpy()
    common_seconds = prep_seconds+compression_seconds+reference_seconds+validation_seconds
    models, rows, tuning_rows = {}, [], []
    for family in protocol['families']:
        started = perf_counter()
        candidates = []
        if family in ('CSM1NN', 'KNN'):
            ks = [1] if family == 'CSM1NN' else [k for k in [3, 5, 9] if k <= m]
            for k in ks:
                predict = lambda z, k=k: nearest_predict(zr, yr, z, k=k)[0]
                candidate_record = {'name': f'k={k}', 'predict': predict,
                                    'weights': None, 'k': k, 'metric_success': True}
                candidate_record['validation_mse'] = float(np.mean((predict(z_val)-yv)**2))
                candidates.append(candidate_record)
        elif family == 'Learned1NN':
            for lam in [0.0, 0.0001, 0.01, 0.1]:
                fitted = fit_metric(zr, yr, lam=lam, max_pairs=2000, seed=seed+30000)
                weights = fitted['weights']
                predict = lambda z, w=weights: nearest_predict(zr, yr, z, weights=w)[0]
                candidates.append({'name': f'lambda={lam:g}', 'predict': predict,
                                   'weights': weights, 'k': 1,
                                   'metric_success': fitted['success'],
                                   'metric_diagnostics': fitted,
                                   'validation_mse': float(np.mean((predict(z_val)-yv)**2))})
        else:
            for setting, estimator in _regression_candidates(family, seed):
                estimator.fit(zr, yr)
                prediction = estimator.predict(z_val)
                candidates.append({'name': setting,
                                   'predict': lambda z, model=estimator: model.predict(z),
                                   'weights': None, 'k': None, 'metric_success': True,
                                   'validation_mse': float(np.mean((prediction-yv)**2))})
        best = min(candidates, key=lambda c: c['validation_mse'])
        tuning_seconds = perf_counter()-started
        best['tuning_seconds'] = tuning_seconds
        models[family] = best
        for c in candidates:
            tuning_rows.append({'compressor': compressor, 'm': m, 'seed': seed,
                                'family': family, 'setting': c['name'],
                                'validation_rmse': np.sqrt(c['validation_mse']),
                                'selected': c is best, 'metric_success': c['metric_success']})
        rows.append({'compressor': compressor, 'm': m, 'seed': seed, 'family': family,
                     'setting': best['name'], 'validation_rmse': np.sqrt(best['validation_mse']),
                     'reference_paths': work['reference_paths'],
                     'reference_work': work['reference_work'],
                     'validation_work': work['validation_work'],
                     'pipeline_work': work['pipeline_work'],
                     'unused_work': work['pipeline_unused'],
                     'prep_s': prep_seconds, 'compress_s': compression_seconds,
                     'reference_sim_s': reference_seconds, 'validation_sim_s': validation_seconds,
                     'fit_and_select_s': tuning_seconds,
                     'development_s': common_seconds+tuning_seconds,
                     'metric_success': best['metric_success']})
    return {'rows': rows, 'tuning_rows': tuning_rows, 'models': models,
            'parts': parts, 'scaler': scaler, 'Zref': zr, 'yref': yr,
            'representatives': representatives, 'reference_labels': reference_labels,
            'validation': validation, 'validation_labels': validation_labels,
            'work': work, 'common_seconds': common_seconds,
            'm': m, 'compressor': compressor, 'seed': seed}


def run_development(df, protocol):
    """Shared experiment runtime and reconstructed standalone pipeline costs differ."""
    if len(df) != protocol['n_contracts']:
        raise ValueError('Protocol contract count does not match the supplied portfolio.')
    if float(protocol['risk_free_rate']) != RISK_FREE_RATE:
        raise ValueError('Update the pricing model and protocol together before changing r.')
    if protocol['label_target'] != 'normalized_gross_GMAB_top_up_at_fixed_fee':
        raise ValueError('The configured label target differs from the implemented model.')
    started = perf_counter()
    states = []
    for seed in protocol['seeds']:
        for m in protocol['representative_counts']:
            for compressor in protocol['compressors']:
                states.append(develop_one(df, protocol, int(m), compressor, int(seed)))
    table = pd.DataFrame([row for state in states for row in state['rows']])
    tuning = pd.DataFrame([row for state in states for row in state['tuning_rows']])
    return states, table, tuning, perf_counter()-started


def freeze_manifest(protocol, development_table):
    """Record all candidate rules and validation-selected settings before audit."""
    simple_settings = development_table[['compressor', 'm', 'seed', 'family', 'setting']]
    manifest = {'protocol': json.loads(json.dumps(protocol)),
                'selected_settings': simple_settings.to_dict(orient='records'),
                'hyperparameter_grids': {
                    'KNN_k': [3, 5, 9], 'Ridge_alpha': [0.1, 1.0, 10.0],
                    'SplineGAM_knots': 4, 'SplineGAM_alpha': [0.1, 1.0, 10.0],
                    'RF_leaf': [2, 5], 'RF_trees': 100,
                    'GB_trees': 100, 'GB_depth': 2, 'GB_learning_rate': 0.05,
                    'metric_lambda': [0.0, 0.0001, 0.01, 0.1], 'metric_max_pairs': 2000},
                'test_used_for_selection': False,
                'features': list(FEATURES),
                'metrics': ['test_rmse', 'test_mae', 'test_p95_abs', 'test_bias',
                            'test_subportfolio_relative_error', 'full_portfolio_relative_error',
                            'end_to_end_s', 'simulation_work'],
                'software': {'python': platform.python_version(), 'numpy': np.__version__,
                             'pandas': pd.__version__, 'scipy': scipy.__version__,
                             'scikit_learn': sklearn.__version__}}
    canonical = json.dumps(manifest, sort_keys=True, separators=(',', ':')).encode()
    manifest['sha256'] = hashlib.sha256(canonical).hexdigest()
    return manifest


def verify_manifest(manifest, protocol, development_table):
    payload = {k: v for k, v in manifest.items() if k != 'sha256'}
    canonical = json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()
    if hashlib.sha256(canonical).hexdigest() != manifest.get('sha256'):
        raise ValueError('The stored manifest content no longer matches its digest.')
    current = freeze_manifest(protocol, development_table)
    if current['sha256'] != manifest['sha256']:
        raise ValueError('Protocol or selected settings changed after the freeze.')


def final_audit(df, protocol, states, development_table, manifest):
    """Construct ALL predictions before evaluating the closed-form test target.

    Full-portfolio predictions retain simulated reference values at reference
    rows, then use each query's premium for dollar totals. The held-out test
    split contains no reference contracts. Simulated validation labels are
    not added to the reference training set after tuning.
    """
    verify_manifest(manifest, protocol, development_table)
    predicted = []
    for state in states:
        start = perf_counter()
        z_full = state['scaler'].transform(df[FEATURES])
        transform_s = perf_counter()-start
        rep_pos = df.index.get_indexer(state['representatives'].index)
        test_pos = df.index.get_indexer(state['parts']['test'].index)
        for family, fitted in state['models'].items():
            start = perf_counter()
            yhat = np.asarray(fitted['predict'](z_full), dtype=float)
            yhat[rep_pos] = state['yref']
            predict_s = perf_counter()-start
            row = {'compressor': state['compressor'], 'm': state['m'],
                   'seed': state['seed'], 'family': family, 'setting': fitted['name'],
                   'simulation_work': state['work']['pipeline_work'],
                   'validation_work': state['work']['validation_work'],
                   'reference_paths': state['work']['reference_paths'],
                   'end_to_end_s': state['common_seconds']+fitted['tuning_seconds']+transform_s+predict_s,
                   'final_prediction_s': transform_s+predict_s,
                   'metric_success': fitted['metric_success']}
            neighbor_position = None
            if family in ('CSM1NN', 'Learned1NN'):
                _, ni = nearest_predict(state['Zref'], state['yref'], z_full[test_pos],
                                        weights=fitted['weights'], k=1)
                neighbor_position = rep_pos[ni[:, 0]]
            predicted.append((row, yhat, test_pos, neighbor_position))
    # Direct MC simulates all N contracts at equal total path work, without tuning.
    direct_rows = []
    for seed in protocol['seeds']:
        n_paths = int(protocol['budget'])//len(df)
        start = perf_counter()
        labels = gmab_mc(df, n_paths, seed=int(seed)+90000)
        direct_seconds = perf_counter()-start
        test_pos = df.index.get_indexer(split_portfolio(df, int(protocol['split_seed']))['test'].index)
        direct_rows.append(({'compressor': 'none', 'm': len(df), 'seed': int(seed),
                             'family': 'MC-all', 'setting': f'{n_paths} paths/contract',
                             'simulation_work': n_paths*len(df), 'validation_work': 0,
                             'reference_paths': n_paths, 'end_to_end_s': direct_seconds,
                             'final_prediction_s': 0.0, 'metric_success': True},
                            labels['value'].to_numpy(), test_pos, None))
    predicted.extend(direct_rows)
    # The first access to final test values occurs HERE, after prediction construction.
    start = perf_counter()
    truth = gmab_exact(df)
    analytic_seconds = perf_counter()-start
    p = df['premium'].to_numpy()
    rows, prediction_rows = [], []
    for row, yhat, test_pos, neighbors in predicted:
        test_scores = score_predictions(truth[test_pos], yhat[test_pos], p[test_pos])
        for key in ['rmse', 'mae', 'p95_abs', 'bias', 'negative_fraction']:
            row['test_'+key] = test_scores[key]
        row['test_subportfolio_relative_error'] = test_scores['portfolio_relative_error']
        row['full_portfolio_relative_error'] = score_predictions(truth, yhat, p)['portfolio_relative_error']
        row['oracle_neighbor_gap_mae'] = (float(np.mean(np.abs(truth[test_pos]-truth[neighbors])))
                                          if neighbors is not None else np.nan)
        rows.append(row)
        prediction_rows.append(pd.DataFrame({
            'contract_id': df.iloc[test_pos]['contract_id'].to_numpy(),
            'compressor': row['compressor'], 'm': row['m'], 'seed': row['seed'],
            'family': row['family'], 'truth': truth[test_pos], 'prediction': yhat[test_pos],
            'premium': p[test_pos]}))
    return pd.DataFrame(rows), pd.concat(prediction_rows, ignore_index=True), {
        'analytic_audit_s': analytic_seconds, 'truth_full': truth,
        'audit_used_for_selection': False, 'analytic_available_for_this_teaching_model': True}


## 3. Configure the whole experiment before running it

The classroom default contains 1,200 contracts, two representative counts, two
compressors and two random seeds. The 10,000-contract configuration below is optional.
The contract split is fixed across repetitions; changing a repetition seed changes
representative selection and simulation/model randomness. Therefore between-seed
variation measures algorithmic variability on this portfolio, not population-wide
uncertainty over all possible insurance portfolios.

All numerical features are scaled using the candidate pool only. The 40 labeled
validation contracts are selected from the validation split without looking at prices.
The remaining validation contracts are not secretly added to the training set.

The small hyperparameter grids are explicit in the helper code. Their computational
cost is charged to the corresponding predictor family. The returned setting is chosen
by **simulated validation MSE**, never by an analytic label.

In [ ]:
PROTOCOL = json.loads(json.dumps(DEFAULT_PROTOCOL))
RUN_RESEARCH_SCALE = False
if RUN_RESEARCH_SCALE:
    PROTOCOL.update(n_contracts=10000, representative_counts=[100, 200],
                    seeds=[2027, 2028, 2029, 2030, 2031], budget=2_000_000,
                    n_validation_labels=100, validation_paths=2048)

portfolio = make_portfolio(PROTOCOL['n_contracts'], PROTOCOL['portfolio_seed'])
parts = split_portfolio(portfolio, PROTOCOL['split_seed'])
split_table = pd.DataFrame({name: {
    'contracts': len(frame), 'labels_used_in_development':
    ('selected representatives' if name == 'candidate' else
     PROTOCOL['n_validation_labels'] if name == 'validation' else 0)
} for name, frame in parts.items()}).T
display(split_table)

budget_rows = []
for m in PROTOCOL['representative_counts']:
    allocation = budget_plan(len(portfolio), m, PROTOCOL['budget'],
                             PROTOCOL['n_validation_labels'], PROTOCOL['validation_paths'])
    budget_rows.append({'representatives': m, **allocation})
budget_table = pd.DataFrame(budget_rows)
display(budget_table)
assert (budget_table['pipeline_work'] <= PROTOCOL['budget']).all()
assert (budget_table['direct_work'] <= PROTOCOL['budget']).all()

### Worked budget example and a question

Under the default budget $B=262{,}144$, validation uses $40\times1024=40{,}960$ paths.
The remaining $221{,}184$ paths are divided among representatives. The direct-all
baseline receives 218 paths per contract, for 261,600 paths in total.

Before looking at any output, explain why 80 representatives should not receive the
same per-representative path count as 40 representatives in this experiment.
Also explain why equal path work need not imply equal wall-clock time.

## 4. Run the development comparison

Each compressor/count/seed condition creates one reference label set. Every predictor
in that condition receives exactly those labels. The simulator uses contract-specific
random streams, so ordering a subset differently does not change a contract's label.

Validation labels are statistically independent of reference labels. Reusing the same
validation labels across model candidates is deliberate: it reduces irrelevant
comparison noise, but the minimum validation error is still optimistic after selection.
It is not a substitute for a held-out test result.

No true test value is computed below. Development may inspect candidate and validation
features, simulated labels, validation errors, and runtimes.

In [ ]:
states, development, tuning, shared_elapsed = run_development(portfolio, PROTOCOL)
validation_summary = (development.groupby(['compressor', 'm', 'family'], as_index=False)
                      .agg(validation_rmse_mean=('validation_rmse', 'mean'),
                           validation_rmse_sd=('validation_rmse', 'std'),
                           development_seconds_mean=('development_s', 'mean')))
display(validation_summary.sort_values('validation_rmse_mean'))
display(tuning[['compressor', 'm', 'seed', 'family', 'setting', 'selected']].head(18))
print(f'Actual runtime of this shared development experiment: {shared_elapsed:.3f} seconds')
print('Final test values have not been evaluated.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for family, group in validation_summary.groupby('family'):
    ax.scatter(group['development_seconds_mean'], 100*group['validation_rmse_mean'],
               label=family, s=45)
ax.set(xlabel='Standalone development cost (seconds)',
       ylabel='Validation RMSE (percentage points of premium)',
       title='Development diagnostics only — final test remains closed')
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
plt.show()

## 5. A cost ledger makes the conclusion assessable

For one standalone pipeline, report

$$t_{\mathrm{total}}=t_{\mathrm{preprocess}}+t_{\mathrm{compress}}
+t_{\mathrm{reference\ MC}}+t_{\mathrm{validation\ MC}}
+t_{\mathrm{fit\ and\ select}}+t_{\mathrm{final\ predict}}.$$

The last term is added in Week 8. The table below includes each earlier term. In the
shared experiment, reference labels are reused across predictors; for a standalone
method they would still need to be generated. Consequently, the sum of the standalone
row costs is not the actual time spent running the shared experiment.

The budget refers to **one compared pipeline**. Exploring multiple representative
counts or compressors costs additional development work. Record the entire study's
runtime separately; do not claim that a search over all pipelines cost only one $B$.

The direct-all benchmark uses the same budget in Week 8. A simulator can outperform
all the learned methods in this simple model. That is a valid finding.

In [ ]:
ledger_columns = ['compressor', 'm', 'seed', 'family', 'reference_paths',
                  'reference_work', 'validation_work', 'pipeline_work', 'unused_work',
                  'prep_s', 'compress_s', 'reference_sim_s', 'validation_sim_s',
                  'fit_and_select_s', 'development_s']
display(development[ledger_columns].head(14))
print('Metric optimizer failures (reported with Euclidean fallback):',
      int((~development['metric_success']).sum()))
print('All pricing/validation labels were generated by Monte Carlo.')

## 6. Freeze the analysis before opening the test set

A freeze is a scientific record, not a security mechanism. Write down the precise
question, model families, tuning grids, feature definitions, representative counts,
budgets, seeds, metrics and intended plots. A cryptographic digest makes accidental
changes detectable. Do not choose a new feature or extension after studying test errors.

The core comparison itself is the research project; an additional extension is optional.
One manageable extension would change the total budget while holding the method menu
fixed. Select it now using validation evidence and include it in the frozen record.
The reference implementation keeps `extension='none'` and already compares two values
of $m$. A post-test idea belongs in future work or a new independent experiment.

Week 8 contains the same default configuration and all helper code. For the team's
actual study, carry the agreed configuration forward before running the test cells.
No earlier output file is required to run the supplied examples.

In [ ]:
manifest = freeze_manifest(PROTOCOL, development)
verify_manifest(manifest, PROTOCOL, development)
OUT = Path('outputs/week07')
OUT.mkdir(parents=True, exist_ok=True)
development.to_csv(OUT/'development_per_run.csv', index=False)
validation_summary.to_csv(OUT/'development_summary.csv', index=False)
tuning.to_csv(OUT/'tuning_record.csv', index=False)
budget_table.to_csv(OUT/'budget_ledger.csv', index=False)
(OUT/'frozen_protocol.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Freeze digest:', manifest['sha256'])
print('Development files written to', OUT.resolve())

## Interpretation and meeting discussion

* Which comparison isolates the learned distance, and which changes representative coverage?
* Does a smaller validation error exceed the visible between-seed variation and label noise?
* Which methods spend most time fitting rather than simulating?
* Did the metric optimizer succeed? A fallback must not be presented as a learned result.
* How many simulated contracts and how much work did the **whole development study** use?

There is no required winner. A useful result can identify a failure mode, explain why
a simple method is competitive, or show that the available label budget is insufficient.

## Reading assignment

**Required:** Feng, Tan and Zheng (2020), *Efficient Simulation Designs for Valuation of
Large Variable Annuity Portfolios*, Section 6.4 and the discussion surrounding the CSM
predictor in Section 5. [Author-hosted paper](https://www.math.uwaterloo.ca/~mbfeng/papers/2020_VAPort.pdf).

Read for experimental reasoning rather than reproduce the published tables. Answer:

1. Why can aggregate portfolio error conceal poor contract-level predictions?
2. What computations should count when two methods claim an efficiency difference?
3. How does our simplified terminal GMAB model differ from the paper's valuation setting?

**Optional:** Section 4 on simulation allocation. Locate the quantity its allocation
optimizes; do not assume that allocating paths for an unweighted sum of representatives
is automatically optimal for a cluster-weighted full portfolio.

## Coding and research homework

### Exercise 1 — Check the budget algebra

For $m\in\{20,40,80,160\}$, produce a table of reference paths, total work and unused
remainder under the classroom budget. Verify the inequalities in code. Explain the
coverage-versus-noise tradeoff without asserting a winner.

In [ ]:
# Starter: extend this list and display the complete budget table.
homework_m = [20, 40]
homework_budget_rows = [dict(m=m, **budget_plan(len(portfolio), m, PROTOCOL['budget'],
                      PROTOCOL['n_validation_labels'], PROTOCOL['validation_paths']))
                        for m in homework_m]
display(pd.DataFrame(homework_budget_rows))

### Exercise 2 — Audit the comparison

Choose one compressor/count/seed condition. Show that every predictor received the
same reference labels and simulation work. Compare the cost of evaluating one fixed
ridge model with the supplied three-alpha selection procedure. Include selection
overhead in your explanation; do not access the test split.

In [ ]:
chosen_state = states[0]
audit_reference_table = chosen_state['representatives'][['contract_id']].copy()
audit_reference_table['simulated_value'] = chosen_state['reference_labels']['value']
audit_reference_table['simulation_se'] = chosen_state['reference_labels']['se']
display(audit_reference_table.head())
# Add your one-alpha versus three-alpha timing comparison here.

### Exercise 3 — Write the research freeze

Produce a 150–250-word plan stating the primary comparison, anticipated outputs,
simulation budget and one possible explanation for each of improvement, no difference,
and deterioration. Decide whether to keep the core study or add one extension. Record
the decision and freeze it before Week 8. Do not write the eventual result in advance.

## Weekly deliverable checklist

- [ ] Contract split and budgets are recorded; test labels have not been inspected.
- [ ] Validation table compares identical labels within each controlled condition.
- [ ] Both standalone costs and the shared experiment runtime are labeled accurately.
- [ ] Optimizer failures and negative value predictions are retained, not hidden.
- [ ] Reading answers, homework code, and a dated freeze manifest are submitted.

**Next week:** execute the frozen analysis, open the test audit, and write what the data support.